In [ ]:
import gdown
import pandas as pd
import numpy as np
import math
import random
import gc
import os
from collections import defaultdict, Counter
from tqdm import tqdm
from sklearn.ensemble import RandomForestClassifier

# -----------------------------------------------------------------------------
# 0. 초기화 및 데이터 다운로드
# -----------------------------------------------------------------------------
# 1. zip 파일의 ID 입력
zip_file_id = 'YOUR_DRIVE_FILE_ID'
output_filename = 'data.zip'

# 2. zip 파일 다운로드
url = f'https://drive.google.com/uc?id={zip_file_id}'
gdown.download(url, output_filename, quiet=False)

# 3. 압축 풀기 (리눅스 명령어를 사용합니다)
# -o: 덮어쓰기 허용, -d: 압축 풀 폴더 지정
!unzip -o data.zip

# 4. 확인
print("파일 목록:", os.listdir())
try:
    train = pd.read_csv('data/train.csv', encoding='utf-8-sig')
    test = pd.read_csv('data/test.csv', encoding='utf-8-sig')
    sample_submission = pd.read_csv('data/sample_submission.csv', encoding='utf-8-sig')
except FileNotFoundError:
    print("파일을 업로드해주세요.")
    exit()

# -----------------------------------------------------------------------------
# 1. 자모 유틸리티
# -----------------------------------------------------------------------------
BASE_CODE = 0xAC00
CHOSUNG_LIST = ['ㄱ','ㄲ','ㄴ','ㄷ','ㄸ','ㄹ','ㅁ','ㅂ','ㅃ','ㅅ','ㅆ','ㅇ','ㅈ','ㅉ','ㅊ','ㅋ','ㅌ','ㅍ','ㅎ']
JUNGSUNG_LIST = ['ㅏ','ㅐ','ㅑ','ㅒ','ㅓ','ㅔ','ㅕ','ㅖ','ㅗ','ㅘ','ㅙ','ㅚ','ㅛ','ㅜ','ㅝ','ㅞ','ㅟ','ㅠ','ㅡ','ㅢ','ㅣ']
JONGSUNG_LIST = ['', 'ㄱ','ㄲ','ㄳ','ㄴ','ㄵ','ㄶ','ㄷ','ㄹ','ㄺ','ㄻ','ㄼ','ㄽ','ㄾ','ㄿ','ㅀ','ㅁ','ㅂ','ㅄ','ㅅ','ㅆ','ㅇ','ㅈ','ㅊ','ㅋ','ㅌ','ㅍ','ㅎ']

def to_jamo_list(text):
    result = []
    if not isinstance(text, str): return []
    for ch in text:
        code = ord(ch) - BASE_CODE
        if code < 0 or code > (0xD7A3 - 0xAC00):
            result.append(ch)
        else:
            cho = code // 588
            jung = (code % 588) // 28
            jong = (code % 588) % 28
            result.extend([CHOSUNG_LIST[cho], JUNGSUNG_LIST[jung], JONGSUNG_LIST[jong]])
    return result

def from_jamo_list(jamo_list):
    result = []
    i = 0
    while i < len(jamo_list) - 2:
        cho, jung, jong = jamo_list[i], jamo_list[i+1], jamo_list[i+2]
        if cho in CHOSUNG_LIST and jung in JUNGSUNG_LIST and jong in JONGSUNG_LIST:
            code = 0xAC00 + (CHOSUNG_LIST.index(cho)*588) + (JUNGSUNG_LIST.index(jung)*28) + JONGSUNG_LIST.index(jong)
            result.append(chr(code))
            i += 3
        else:
            result.append(cho)
            i += 1
    return "".join(result)

def align_jamo_list(src_list, tgt_list):
    n, m = len(src_list), len(tgt_list)
    dp = [[0]*(m+1) for _ in range(n+1)]
    for i in range(1, n+1): dp[i][0] = i
    for j in range(1, m+1): dp[0][j] = j
    for i in range(1, n+1):
        for j in range(1, m+1):
            cost = 0 if src_list[i-1] == tgt_list[j-1] else 1
            dp[i][j] = min(dp[i-1][j]+1, dp[i][j-1]+1, dp[i-1][j-1]+cost)
    alignment = []
    i, j = n, m
    while i>0 or j>0:
        if i>0 and dp[i][j] == dp[i-1][j] + 1:
            alignment.append((src_list[i-1], '<DEL>')); i-=1
        elif j>0 and dp[i][j] == dp[i][j-1] + 1:
            alignment.append((None, tgt_list[j-1])); j-=1
        else:
            alignment.append((src_list[i-1], tgt_list[j-1])); i-=1; j-=1
    return alignment[::-1]

def process_chunk_to_numpy(df_chunk, token_to_id, window_size=7):
    data_X, data_y = [], []
    PAD = '<PAD>'
    pad_id = token_to_id.get(PAD, 0)
    
    for idx, row in df_chunk.iterrows():
        in_jamos = to_jamo_list(row['input'])
        out_jamos = to_jamo_list(row['output'])
        alignment = align_jamo_list(in_jamos, out_jamos)
        
        in_ids = [token_to_id.get(t, 0) for t in in_jamos]
        padded_ids = [pad_id]*(window_size//2) + in_ids + [pad_id]*(window_size//2)
        
        curr = 0
        for src, tgt in alignment:
            if src is None: continue
            win = padded_ids[curr : curr + window_size]
            tgt_id = token_to_id.get(tgt, 0)
            data_X.append(win)
            data_y.append(tgt_id)
            curr += 1
    return np.array(data_X, dtype=np.int16), np.array(data_y, dtype=np.int16)

# -----------------------------------------------------------------------------
# 2. [필수] 전체 토큰 맵 구축 (메모리 절약형 스캔)
# -----------------------------------------------------------------------------
print("1단계: 전체 데이터 스캔하여 토큰 맵 구축 중...")
token_set = set(['<PAD>', '<DEL>'])
vocab_set = set()

# 청크 단위로 읽어서 토큰만 수집 (RAM 1GB 이하 사용)
for chunk in pd.read_csv('data/train.csv', chunksize=5000, encoding='utf-8-sig'):
    # Input과 Output의 모든 글자를 수집해야 함
    text_blob = "".join(chunk['input'].astype(str).tolist()) + "".join(chunk['output'].astype(str).tolist())
    token_set.update(to_jamo_list(text_blob))
    for txt in chunk['output'].astype(str):
        for w in txt.split(): vocab_set.add(w)

token_to_id = {t: i for i, t in enumerate(sorted(list(token_set)))}
id_to_token = {i: t for t, i in token_to_id.items()}
print(f"토큰 수: {len(token_to_id)}, 단어 사전 수: {len(vocab_set)}")
gc.collect()

# =============================================================================
# 3. [수정됨] RandomForest 점진적 학습 (모델 강제 초기화 포함)
# =============================================================================
print("2단계: RandomForest 스트리밍 학습 시작...")

# [안전장치] 이전 실행에서 남은 오염된 모델이 있다면 삭제
if 'rf_model' in globals():
    del rf_model
    gc.collect()
    print("이전 모델 기록 삭제 완료 (메모리 초기화)")

# 1. 모델 초기화 (n_estimators=1로 시작)
# 따뜻한 출발(warm_start)을 하되, 아주 깨끗한 상태에서 시작합니다.
rf_model = RandomForestClassifier(
    n_estimators=1,      # 1개로 시작
    n_jobs=1,            # 1로 고정 (메모리 방어)
    max_depth=20,
    warm_start=True,     # 점진적 학습
    random_state=42,
    verbose=0
)

# 2. [핵심] 클래스 사전 교육 (Calibration)
# 모델에게 "세상에는 총 225개의 자모가 있다"는 것을 미리 알려줍니다.
n_classes = len(token_to_id)
pad_id = token_to_id.get('<PAD>', 0)

# 모든 클래스(0 ~ n_classes-1)를 포함한 아주 작은 더미 데이터 생성
X_dummy = np.full((n_classes, 7), pad_id, dtype=np.int16)
y_dummy = np.arange(n_classes, dtype=np.int16)

print(f"클래스 사전 등록 중... (총 {n_classes}개 자모)")
rf_model.fit(X_dummy, y_dummy) # <--- 여기서 모델의 정답지 개수가 225개로 고정됨!

print("모델 구조 확정 완료. 실제 데이터 학습 시작...")

# 3. 실제 데이터 학습 루프
# 3. 실제 데이터 학습 루프
CHUNK_SIZE = 2000
total_rows = 11263
n_chunks = math.ceil(total_rows / CHUNK_SIZE)
trees_per_chunk = max(1, 150 // n_chunks) # 목표 150그루

chunk_iter = pd.read_csv('data/train.csv', chunksize=CHUNK_SIZE, encoding='utf-8-sig')

for i, chunk in enumerate(tqdm(chunk_iter, total=n_chunks, desc="Training")):
    # 데이터 변환
    X_chunk, y_chunk = process_chunk_to_numpy(chunk, token_to_id, window_size=7)
    
    if len(X_chunk) == 0: continue
    
    # 나무 추가 (기존 개수 + 추가분)
    rf_model.n_estimators += trees_per_chunk
    
    # -------------------------------------------------------------------------
    # [FIX] 클래스 누락 방지: 더미 데이터(모든 클래스 포함)를 현재 청크와 병합
    # -------------------------------------------------------------------------
    X_train = np.vstack([X_chunk, X_dummy])
    y_train = np.concatenate([y_chunk, y_dummy])
    
    rf_model.fit(X_train, y_train)
    
    # 메모리 정리
    del X_chunk, y_chunk, chunk, X_train, y_train
    gc.collect()

print(f"학습 완료 (총 나무 개수: {rf_model.n_estimators})")

# -----------------------------------------------------------------------------
# 4. 언어 모델 & 디코더 & 추론
# -----------------------------------------------------------------------------
def build_language_model_stream(csv_path):
    print("3단계: 언어 모델(LM) 구축 중...")
    lm_counter = Counter()
    for chunk in pd.read_csv(csv_path, usecols=['output'], chunksize=10000, encoding='utf-8-sig'):
        for txt in chunk['output']:
            if not isinstance(txt, str): continue
            jamos = [(None)] + to_jamo_list(txt)
            for i in range(len(jamos)-1):
                lm_counter[(jamos[i], jamos[i+1])] += 1
    lm_prob = defaultdict(lambda: 1e-7)
    total_counts = defaultdict(int)
    for (prev, curr), count in lm_counter.items():
        total_counts[prev] += count
    for (prev, curr), count in lm_counter.items():
        lm_prob[(prev, curr)] = count / total_counts[prev]
    return lm_prob

lm_prob = build_language_model_stream('data/train.csv')

def decode_hybrid(text, model, token_to_id, id_to_token, classes, lm_prob, vocab_set, 
                  window_size=7, beam_size=10, lm_weight=2.5, threshold=3.0):
    if not isinstance(text, str): return ""
    in_jamos = to_jamo_list(text)
    pad_id = token_to_id.get('<PAD>', 0)
    
    in_ids = [token_to_id.get(t, 0) for t in in_jamos]
    padded_ids = [pad_id]*(window_size//2) + in_ids + [pad_id]*(window_size//2)
    
    X_enc = []
    for i in range(len(in_jamos)):
        X_enc.append(padded_ids[i : i + window_size])
    
    if not X_enc: return text
    probs = model.predict_proba(X_enc)
    
    beam = [([], None, 0.0)] 
    for i in range(len(in_jamos)):
        curr_probs = probs[i]
        top_k_indices = np.argsort(curr_probs)[-beam_size:] 
        new_beam = []
        for (decoded, last_char, score) in beam:
            for idx in top_k_indices:
                char_id = classes[idx]
                char_prob = curr_probs[idx]
                pred_char = id_to_token[char_id]
                
                ml_score = math.log(char_prob + 1e-9)
                lm_p = lm_prob.get((last_char, pred_char), 1e-7)
                lm_score = math.log(lm_p)
                total_score = score + ml_score + (lm_weight * lm_score)
                
                if pred_char == '<DEL>': new_beam.append((decoded, last_char, total_score))
                else: new_beam.append((decoded + [pred_char], pred_char, total_score))
        new_beam.sort(key=lambda x: x[2], reverse=True)
        beam = new_beam[:beam_size]
    
    candidates = []
    for (decoded, _, sc) in beam:
        word = from_jamo_list(decoded)
        candidates.append((word, sc))
        
    if not candidates: return text
    best_word, best_score = candidates[0]
    if best_word in vocab_set: return best_word
    for i in range(1, len(candidates)):
        cand_word, cand_score = candidates[i]
        if best_score - cand_score > threshold: break
        if cand_word in vocab_set: return cand_word
    return best_word

print("4단계: 최종 추론 시작...")

outputs = []
rf_classes = rf_model.classes_

for idx, row in tqdm(test.iterrows(), total=len(test)):
    obf_text = row['input']
    tokens = obf_text.split()
    new_tokens = []
    for t in tokens:
        restored = decode_hybrid(t, rf_model, token_to_id, id_to_token, rf_classes, lm_prob, vocab_set,
                                 window_size=7, beam_size=10, lm_weight=2.5, threshold=3.0)
        new_tokens.append(restored)
    outputs.append(" ".join(new_tokens))
    if idx % 100 == 0: gc.collect()

sample_submission['output'] = outputs
sample_submission.to_csv('submission_final_v5.csv', index=False)
print("모든 작업 완료! submission_final_v5.csv 저장됨.")